# Flow capacity vs. derived-tail collapse — GPU sweep

Does scaling the flow recover the tail that an equation-**derived** integrator clips,
or does it plateau far below what max-entropy wants?

**Setup.** An `EquationEstimate` like
`anomaly = trend + 0.075*enso_oni - volcanic ~ N(0, 0.035)` turns `anomaly` into a
derived flow coordinate. The `eqn_dist` loss pins only the residual's first two
moments (mean 0, var σ²), **not** its independence from the right-hand side. A
finite flow satisfies those two moments while anti-correlating the residual with
the RHS in the tail — clipping `P(anomaly > 1.60)` to ≈0, even though the
max-entropy solution (independent, *same marginals*) has a ~0.06–0.09 tail **and**
higher entropy. Entropy accounting showed the clip costs the flow ~0.06 nats, i.e.
it's a dominated solution — so the question is whether it's a **capacity wall**
(tail climbs to the max-ent target as params grow) or the **parameterization**
(plateaus regardless of size → the derived-readout reparam is the real fix).

Per flow size we log: `P_flow` (derived `P(anomaly>1.60)`), `P_maxent`
(reconvolving the flow's *own* leaf marginals independently — the tail the
objective wants), and `I` = total correlation among `(trend, oni, volc, r)` =
the entropy (nats) the flow sacrificed.

> **Prerequisite:** the equation-language changes (`EquationEstimate`,
> `maxent_sampler/equations.py`, the `eqn_det`/`eqn_dist` kinds) must be committed
> to the branch cloned below. The setup cell asserts they're importable.

In [ ]:
# --- Setup: clone repo, install the deps Colab lacks, keep Colab's own jax ---
import os, sys, subprocess

REPO = "/content/calibrated_response"
BRANCH = "main"   # set to the branch carrying the equation-language changes
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/amdson/calibrated_response.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)

# optax/jaxopt are the only solver deps Colab doesn't ship; installing the
# package itself would drag in pinned jax/jaxlib and clobber the GPU build.
%pip -q install optax jaxopt

import jax
jax.config.update("jax_compilation_cache_dir", f"{REPO}/.jax_cache")
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
print("jax backend:", jax.default_backend(), jax.devices())
assert jax.default_backend() != "cpu", "No GPU — switch the runtime type first"

# equation language must be present on the cloned branch
from calibrated_response.models.query import EquationEstimate          # noqa: F401
from calibrated_response.maxent_sampler.equations import compile_residual  # noqa: F401
print("equation language present ✓")

In [ ]:
# --- Experiment: the 2026-warmest-year equation model + a fit-and-measure fn ---
import numpy as np
from calibrated_response.models.variable import BinaryVariable, ContinuousVariable
from calibrated_response.models.natural_response import parse_natural_syntax as P
from calibrated_response.maxent_sampler.distribution_builder import DistributionBuilder

RECORD   = 1.60
HAND_MC  = 0.067   # hand Monte-Carlo of the same structural model (reference)

VARIABLES = [
    BinaryVariable(name="record_2026", description="2026 warmest year on record"),
    ContinuousVariable(name="anomaly_2026", description="2026 global anomaly", lower_bound=1.25, upper_bound=1.85, unit="C"),
    ContinuousVariable(name="trend_2026", description="ENSO/volcano-free baseline", lower_bound=1.40, upper_bound=1.66, unit="C"),
    ContinuousVariable(name="enso_oni_2026", description="2026 annual-mean ONI", lower_bound=-2.0, upper_bound=2.0, unit="index"),
    ContinuousVariable(name="volcanic_2026", description="volcanic cooling in 2026", lower_bound=0.0, upper_bound=0.35, unit="C"),
]
EXPRS = [
    "E[trend_2026] = 1.53", "P(trend_2026 > 1.60) < 0.05", "P(trend_2026 < 1.46) < 0.05",
    "E[enso_oni_2026] = -0.2", "P(enso_oni_2026 > 0.5) < 0.18", "P(enso_oni_2026 < -1.1) < 0.10",
    "E[volcanic_2026] = 0.015", "P(volcanic_2026 > 0.10) < 0.06",
    "anomaly_2026 = trend_2026 + 0.075*enso_oni_2026 - volcanic_2026 ~ N(0, 0.035)",
    "record_2026 = ind(anomaly_2026 > 1.60)",
]
ESTS = [P(e) for e in EXPRS]

def fit_and_measure(n_layers, hidden, n_dummy, steps, n_samples, eval_samples, seed):
    b = DistributionBuilder(VARIABLES, ESTS, n_layers=n_layers, hidden=hidden, n_dummy=n_dummy)
    npar = b.model.net.n_params
    b.build(target_variable="record_2026", steps=steps, n_samples=n_samples, seed=seed)
    H = b.entropy(60000)
    x = b.sample_dict(eval_samples, seed=seed + 1)
    tr, on, vo, an = x["trend_2026"], x["enso_oni_2026"], x["volcanic_2026"], x["anomaly_2026"]
    r = an - (tr + 0.075 * on - vo)
    rng = np.random.default_rng(seed)
    recon = (rng.permutation(tr) + 0.075 * rng.permutation(on) - rng.permutation(vo)
             + rng.normal(0, 0.035, len(tr)))
    C = np.corrcoef(np.vstack([tr, on, vo, r]))
    I = float(-0.5 * np.log(max(np.linalg.det(C), 1e-12)))
    return dict(n_layers=n_layers, hidden=hidden, n_dummy=n_dummy, seed=seed,
                params=int(npar), resid_sd=float(r.std()), entropy=float(H),
                p_flow=float(np.mean(an > RECORD)), p_maxent=float(np.mean(recon > RECORD)),
                total_corr=I)

# --- sweep grid (GPU makes the big flows cheap) ---
CONFIGS = [           # (n_layers, hidden, n_dummy)
    (2,  16,  0),
    (4,  32,  0),
    (8,  64,  0),     # current default (~8k params)
    (12, 128, 0),
    (16, 256, 0),
    (20, 384, 0),
    (16, 256, 8),     # + dummy dims: wider conditioning, same objective
    (24, 512, 8),
]
SEEDS        = [0, 1, 2]
STEPS        = 5000
N_SAMPLES    = 6000
EVAL_SAMPLES = 200000
print(f"{len(CONFIGS)} sizes x {len(SEEDS)} seeds = {len(CONFIGS) * len(SEEDS)} fits")

In [ ]:
# --- Run the sweep (resumable: rows append to results/capacity_sweep.jsonl) ---
import json, time
os.makedirs("results", exist_ok=True)
OUT = "results/capacity_sweep.jsonl"

done = set()
if os.path.exists(OUT):
    for line in open(OUT):
        d = json.loads(line)
        done.add((d["n_layers"], d["hidden"], d["n_dummy"], d["seed"]))

t0 = time.time()
with open(OUT, "a") as f:
    for (nl, h, nd) in CONFIGS:
        for s in SEEDS:
            if (nl, h, nd, s) in done:
                continue
            row = fit_and_measure(nl, h, nd, STEPS, N_SAMPLES, EVAL_SAMPLES, s)
            f.write(json.dumps(row) + "\n"); f.flush()
            print(f"L{nl:<2} h{h:<3} d{nd} seed{s} | {row['params']:>7} par | "
                  f"resid_sd={row['resid_sd']:.4f} P_flow={row['p_flow']:.4f} "
                  f"P_maxent={row['p_maxent']:.4f} I={row['total_corr']:.4f} "
                  f"({time.time() - t0:.0f}s)")
print(f"\nDone in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# --- Aggregate + plot: does the derived tail climb to the max-ent target? ---
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

rows = [json.loads(l) for l in open("results/capacity_sweep.jsonl")]
by = defaultdict(list)
for d in rows:
    by[(d["n_layers"], d["hidden"], d["n_dummy"])].append(d)

order = sorted(by, key=lambda k: np.mean([d["params"] for d in by[k]]))
def col(k, key):
    return np.array([d[key] for d in by[k]])
params  = np.array([np.mean(col(k, "params"))     for k in order])
pf_m    = np.array([np.mean(col(k, "p_flow"))     for k in order])
pf_s    = np.array([np.std(col(k, "p_flow"))      for k in order])
pm_m    = np.array([np.mean(col(k, "p_maxent"))   for k in order])
I_m     = np.array([np.mean(col(k, "total_corr")) for k in order])
labels  = [f"L{k[0]} h{k[1]}" + (f" d{k[2]}" if k[2] else "") for k in order]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))
ax1.errorbar(params, pf_m, yerr=pf_s, marker="o", lw=2, capsize=3, label="P_flow (derived)")
ax1.plot(params, pm_m, marker="s", lw=2, ls="--", label="P_maxent (independent target)")
ax1.axhline(HAND_MC, color="gray", ls=":", label=f"hand Monte-Carlo ({HAND_MC})")
ax1.set_xscale("log"); ax1.set_xlabel("flow parameters"); ax1.set_ylabel("P(anomaly > 1.60)")
ax1.set_title("Does capacity buy back the tail?"); ax1.legend(); ax1.set_ylim(bottom=0)

ax2.plot(params, I_m, marker="o", lw=2, color="crimson")
ax2.set_xscale("log"); ax2.set_xlabel("flow parameters")
ax2.set_ylabel("I(trend,oni,volc,r)  [nats sacrificed]")
ax2.set_title("Entropy the clip costs (max-ent wants 0)"); ax2.axhline(0, color="gray", lw=0.7)
for ax in (ax1, ax2):
    for xp, lb in zip(params, labels):
        ax.annotate(lb, (xp, ax.get_ylim()[0]), rotation=90, va="bottom",
                    ha="center", fontsize=7, alpha=0.5)
plt.tight_layout(); plt.savefig("results/capacity_sweep.png", dpi=130); plt.show()

In [ ]:
# --- Download results (merge results/ back into the repo locally) ---
from google.colab import files
import shutil
shutil.make_archive("capacity_sweep_results", "zip", "results")
files.download("capacity_sweep_results.zip")